In [1]:
import polars as pl
import os

from pathlib import Path

In [2]:
DONOR = 'donor_4'

In [3]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined' / DONOR

# Merge the experiment results

In [4]:
# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [5]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_aucprc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",39,71.0,0.78,71.5,0.8058,0.7919,0.6984,0.761,0.7283,0.888
"""LogisticRegression""",1011,"""H3K4me3""",null,69.8,0.7644,71.8,0.7992,0.7801,0.7523,0.6534,0.6994,0.762
"""RandomForest""",1011,"""H3K4me3""",null,70.4,0.7813,72.1,0.8057,0.7925,0.7528,0.6614,0.7041,0.767
"""SVM_Linear""",1011,"""H3K4me3""",null,69.9,0.7636,71.7,0.801,0.7845,0.7506,0.6534,0.6986,0.76
"""DirectRanker""",123,"""H3K4me3""",60,74.1,0.83,71.6,0.8145,0.7966,0.6891,0.7703,0.7274,0.892
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,73.9,0.8033,71.4,0.755,0.7221,0.7315,0.6502,0.6885,0.741
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",80,72.4,0.81,74.9,0.8361,0.8211,0.7386,0.7753,0.7565,0.927
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,69.7,0.7381,71.1,0.7686,0.7437,0.7477,0.6421,0.6909,0.735


In [6]:
# Save the results by model 
MODEL_RESULT_FOLDER = OUTPUT_PATH / 'model_results'
os.makedirs(MODEL_RESULT_FOLDER, exist_ok=True)

for model_name in pl_df["model"].unique():
    pl_df.filter(pl_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}.csv", include_header=True)

In [7]:
# Save the results for all models and all seeds
pl_df.write_csv(OUTPUT_PATH / f"{DONOR}_all_results.csv", include_header=True)

In [8]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
        pl.col("test_aucprc").mean().alias("test_aucprc_mean"),
        pl.col("test_aucprc").std().alias("test_aucprc_std"),
        pl.col("antisymmetry").mean().alias("antisymmetry_mean"),
        pl.col("antisymmetry").std().alias("antisymmetry_std")
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [9]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std,test_aucprc_mean,test_aucprc_std,antisymmetry_mean,antisymmetry_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",73.68,2.7662,0.8187,0.0219,74.94,1.3465,0.8301,0.0154,0.8192,0.0152,0.7786,0.0142
"""H3K27ac""","""SVM_Linear""",72.24,2.5822,0.788,0.0238,73.22,1.4025,0.8022,0.016,0.7813,0.0173,0.7648,0.0118
"""H3K27ac""","""LogisticRegression""",72.48,2.7698,0.7883,0.025,73.2,1.3096,0.802,0.0169,0.7808,0.0193,0.7624,0.0119
"""H3K27ac""","""DirectRanker""",72.36,2.0182,0.814,0.0288,72.88,0.9884,0.8198,0.0112,0.8039,0.0109,0.8636,0.0043
"""H3K27ac-H3K27me3""","""RandomForest""",73.66,2.7144,0.8201,0.0212,75.02,1.6769,0.8315,0.0153,0.8205,0.0145,0.788,0.0088
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",70.72,2.543,0.7633,0.0244,71.26,0.9182,0.7704,0.0168,0.7396,0.0184,0.743,0.0181
"""H3K9me3-H3K27me3""","""RandomForest""",53.62,1.3442,0.5392,0.0088,52.68,1.1925,0.546,0.0074,0.5275,0.008,0.1066,0.0128
"""H3K9me3-H3K27me3""","""DirectRanker""",51.12,1.0941,0.55,0.0071,52.6,0.8944,0.554,0.0188,0.5283,0.0099,0.1348,0.0088


In [10]:
summary_df.write_csv(OUTPUT_PATH/ f"{DONOR}_result_summary.csv", include_header=True)

In [11]:
for model_name in summary_df["model"].unique():
    summary_df.filter(summary_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}_summary.csv", include_header=True)

# Merge the label distribution

In [12]:
# Read all label distribution CSV into single dataframe
label_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-label-distribution.csv")

In [13]:
label_df

seed,histone_marker,split,label,count,total,percentage
i64,str,str,i64,i64,i64,f64
1011,"""H3K4me3""","""Train""",0,4082,8000,51.02
1011,"""H3K4me3""","""Train""",1,3918,8000,48.98
1011,"""H3K4me3""","""Val""",0,518,1000,51.8
1011,"""H3K4me3""","""Val""",1,482,1000,48.2
1011,"""H3K4me3""","""Test""",0,498,1000,49.8
…,…,…,…,…,…,…
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Train""",1,3923,8000,49.04
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",0,529,1000,52.9
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",1,471,1000,47.1


In [14]:
# Make the summary by seed, split and label
label_summary_df = (
    label_df
    .group_by(['split', 'label'])
    .agg(pl.col('percentage').mean().round(2).alias('average_percentage_%'))
    .sort(['split', 'label'])
)

In [15]:
label_summary_df

split,label,average_percentage_%
str,i64,f64
"""Test""",0,50.72
"""Test""",1,49.28
"""Train""",0,50.99
"""Train""",1,49.01
"""Val""",0,52.04
"""Val""",1,47.96
